## FAISS

Faiss (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is developed by Facebook AI Research (FAIR) and is widely used in applications such as information retrieval, natural language processing, and machine learning. FAISS provides various algorithms for indexing and searching large collections of high-dimensional vectors, making it suitable for tasks like nearest neighbor search, clustering, and dimensionality reduction. It supports both CPU and GPU implementations, allowing for efficient processing of large datasets.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data.txt")

documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
docs = text_splitter.split_documents(documents)


In [3]:
embeddings = OllamaEmbeddings(model="embeddinggemma:latest")
db=FAISS.from_documents(docs, embeddings)
db

C:\Users\Deepak Antony\AppData\Local\Temp\ipykernel_8860\1928960311.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="embeddinggemma:latest")


In [11]:
query = "What is the assurance of Tamizhaga Vetri Kazhagam?"
docs = db.similarity_search(query)
docs

[Document(id='0dfda204-154a-4a22-83cf-f2ed878fbc67', metadata={'source': 'data.txt'}, page_content='I wish to assure you at this moment that all members of the Tamilaga Vettri Kazhagam will certainly be helpful to you. The responsibility of protecting the dignity of the members has been entrusted'),
 Document(id='ead6a17e-58ee-4224-b1fd-8a39001aab93', metadata={'source': 'data.txt'}, page_content='members has been entrusted to you. I conclude by expressing my confidence that both of you (the Speaker and Deputy Speaker) will protect this House, ensuring that legislative proceedings are'),
 Document(id='fa3fcd0b-9b2b-4b4a-a526-16455de26379', metadata={'source': 'data.txt'}, page_content='legislative proceedings are conducted efficiently, democratic values and norms are upheld, and the noble quality of respecting fellow human beings is established.'),
 Document(id='08018ca6-eca1-4bfa-8e68-ad080e4f3b5a', metadata={'source': 'data.txt'}, page_content='Therefore, in tradition, whenever a new

In [12]:
docs[0].page_content

'I wish to assure you at this moment that all members of the Tamilaga Vettri Kazhagam will certainly be helpful to you. The responsibility of protecting the dignity of the members has been entrusted'

## Retriever

We can also convert the vector store into retreiver calss. This allows us to easily use it other langchain methods which largely work with retrievers.

In [13]:
retriever = db.as_retriever()
retriever.invoke(query)

[Document(id='0dfda204-154a-4a22-83cf-f2ed878fbc67', metadata={'source': 'data.txt'}, page_content='I wish to assure you at this moment that all members of the Tamilaga Vettri Kazhagam will certainly be helpful to you. The responsibility of protecting the dignity of the members has been entrusted'),
 Document(id='ead6a17e-58ee-4224-b1fd-8a39001aab93', metadata={'source': 'data.txt'}, page_content='members has been entrusted to you. I conclude by expressing my confidence that both of you (the Speaker and Deputy Speaker) will protect this House, ensuring that legislative proceedings are'),
 Document(id='fa3fcd0b-9b2b-4b4a-a526-16455de26379', metadata={'source': 'data.txt'}, page_content='legislative proceedings are conducted efficiently, democratic values and norms are upheld, and the noble quality of respecting fellow human beings is established.'),
 Document(id='08018ca6-eca1-4bfa-8e68-ad080e4f3b5a', metadata={'source': 'data.txt'}, page_content='Therefore, in tradition, whenever a new

### Similarity Search with score

There are some FAISS specific methods like similarity search with score which gives us the similarity score along with the document. This is useful when we want to filter out documents based on a certain threshold.

In [14]:
ss_score = db.similarity_search_with_score(query)
ss_score

[(Document(id='0dfda204-154a-4a22-83cf-f2ed878fbc67', metadata={'source': 'data.txt'}, page_content='I wish to assure you at this moment that all members of the Tamilaga Vettri Kazhagam will certainly be helpful to you. The responsibility of protecting the dignity of the members has been entrusted'),
  np.float32(0.7609712)),
 (Document(id='ead6a17e-58ee-4224-b1fd-8a39001aab93', metadata={'source': 'data.txt'}, page_content='members has been entrusted to you. I conclude by expressing my confidence that both of you (the Speaker and Deputy Speaker) will protect this House, ensuring that legislative proceedings are'),
  np.float32(1.2966608)),
 (Document(id='fa3fcd0b-9b2b-4b4a-a526-16455de26379', metadata={'source': 'data.txt'}, page_content='legislative proceedings are conducted efficiently, democratic values and norms are upheld, and the noble quality of respecting fellow human beings is established.'),
  np.float32(1.4615846)),
 (Document(id='08018ca6-eca1-4bfa-8e68-ad080e4f3b5a', meta

In [15]:
db.save_local("faiss_index")

In [17]:
new_df = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [18]:
docs = new_df.similarity_search(query)

In [19]:
docs

[Document(id='0dfda204-154a-4a22-83cf-f2ed878fbc67', metadata={'source': 'data.txt'}, page_content='I wish to assure you at this moment that all members of the Tamilaga Vettri Kazhagam will certainly be helpful to you. The responsibility of protecting the dignity of the members has been entrusted'),
 Document(id='ead6a17e-58ee-4224-b1fd-8a39001aab93', metadata={'source': 'data.txt'}, page_content='members has been entrusted to you. I conclude by expressing my confidence that both of you (the Speaker and Deputy Speaker) will protect this House, ensuring that legislative proceedings are'),
 Document(id='fa3fcd0b-9b2b-4b4a-a526-16455de26379', metadata={'source': 'data.txt'}, page_content='legislative proceedings are conducted efficiently, democratic values and norms are upheld, and the noble quality of respecting fellow human beings is established.'),
 Document(id='08018ca6-eca1-4bfa-8e68-ad080e4f3b5a', metadata={'source': 'data.txt'}, page_content='Therefore, in tradition, whenever a new